# 3.2 粒子取向分布的影响

**学习目标**
- 理解粒子倾角分布的概率密度函数
- 掌握高斯分布下后向散射协方差矩阵的计算
- 观察倾角标准差对极化变量的影响
- 理解"倾角分散"对相关系数的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 3, pp. 271-310

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 倾角分布

实际大气中粒子并非完全垂直下落，而是有随机摆动。倾角 $\theta$ 的概率密度函数通常用高斯分布描述：

$$p(\theta) = \frac{1}{\sqrt{2\pi}\sigma_\theta} \exp\left(-\frac{\theta^2}{2\sigma_\theta^2}\right)$$

其中 $\sigma_\theta$ 是倾角的标准差。

### 集合平均的后向散射

对于取向随机的粒子集合，后向散射矩阵元素需要对其取向分布积分：

$$\langle S_{hh} \rangle = \int_0^{2\pi} S_{hh}(\theta) p(\theta) d\theta$$

In [ ]:
def gaussian_canting_angle(theta, sigma):
    """高斯倾角分布的概率密度"""
    return np.exp(-theta**2 / (2*sigma**2)) / (sigma * np.sqrt(2*np.pi))

def backscatter_amplitude_rayleigh(D, wavelength, theta, m=1.33+0.001j):
    """
    瑞利散射下单个粒子的后向散射幅度
    D: 直径
    theta: 倾角
    """
    k = 2 * np.pi / wavelength
    K_sq = np.abs((m**2 - 1) / (m**2 + 2))**2
    # 球形粒子的简化散射幅度（实际与倾角无关）
    # 这里用椭球近似说明倾角效应
    S0 = (np.pi**2 * D**3 / (4*wavelength)) * K_sq
    return S0

def covariance_matrix_elements(D, wavelength, sigma_theta, m=1.33+0.001j):
    """
    计算后向散射协方差矩阵元素
    <ShhShh*>, <SvvSvv*>, <ShhSvv*>
    """
    theta_range = np.linspace(-np.pi/2, np.pi/2, 1000)
    p_theta = gaussian_canting_angle(theta_range, sigma_theta)
    
    # 散射幅度（简化模型）
    S_base = backscatter_amplitude_rayleigh(D, wavelength, 0, m)
    
    # 倾角对幅度的影响（简化）
    # 假设水平和垂直偏振的响应与 cos^2(theta) 和 sin^2(theta) 相关
    f_hh = np.cos(theta_range)**2
    f_vv = np.sin(theta_range)**2
    
    S_hh = S_base * f_hh
    S_vv = S_base * f_vv
    
    # 协方差元素
    cov_hh_hh = np.trapezoid(np.abs(S_hh)**2 * p_theta, theta_range)
    cov_vv_vv = np.trapezoid(np.abs(S_vv)**2 * p_theta, theta_range)
    cov_hh_vv = np.trapezoid(S_hh * np.conj(S_vv) * p_theta, theta_range)
    
    return {'ShhShh': cov_hh_hh, 'SvvSvv': cov_vv_vv, 'ShhSvv': cov_hh_vv}

def calculate_rho_hv(cov_hh_vv, cov_hh_hh, cov_vv_vv):
    """计算相关系数 ρhv"""
    denom = np.sqrt(cov_hh_hh * cov_vv_vv)
    return cov_hh_vv / denom if denom > 0 else 0

## 2. 倾角分布可视化

In [ ]:
def plot_orientation_effects(D=0.002, wavelength=0.10, sigma_theta=15.0):
    """可视化取向分布对极化变量的影响"""
    
    # 倾角范围
    theta_range = np.linspace(-90, 90, 180)
    
    # 不同标准差的倾角分布
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 倾角分布
    ax1 = axes[0, 0]
    sigma_values = [5, 15, 30, 45]
    colors = ['blue', 'green', 'orange', 'red']
    for sigma, c in zip(sigma_values, colors):
        p = gaussian_canting_angle(np.radians(theta_range), np.radians(sigma))
        ax1.plot(theta_range, p, color=c, linewidth=2, label=f'σθ={sigma}°')
    ax1.set_xlabel('倾角 θ (degrees)', fontsize=11)
    ax1.set_ylabel('概率密度 p(θ)', fontsize=11)
    ax1.set_title('高斯倾角分布', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: 归一化散射幅度 vs 倾角
    ax2 = axes[0, 1]
    theta_rad = np.radians(theta_range)
    f_hh = np.cos(theta_rad)**2
    f_vv = np.sin(theta_rad)**2
    ax2.plot(theta_range, f_hh, 'b-', linewidth=2, label='|S_{hh}|² (水平)')
    ax2.plot(theta_range, f_vv, 'r-', linewidth=2, label='|S_{vv}|² (垂直)')
    ax2.set_xlabel('倾角 θ (degrees)', fontsize=11)
    ax2.set_ylabel('归一化散射功率', fontsize=11)
    ax2.set_title('散射功率随倾角变化', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3: ρhv vs 倾角标准差
    ax3 = axes[1, 0]
    sigma_range = np.linspace(1, 60, 100)
    rho_hv_values = []
    ZDR_values = []
    for sig in sigma_range:
        cov = covariance_matrix_elements(D, wavelength, np.radians(sig))
        rho = calculate_rho_hv(cov['ShhSvv'], cov['ShhShh'], cov['SvvSvv'])
        rho_hv_values.append(np.abs(rho))
        ZDR_values.append(10*np.log10(cov['ShhShh']/cov['SvvSvv'] + 1e-10))
    ax3.plot(sigma_range, rho_hv_values, 'g-', linewidth=2)
    ax3.set_xlabel('倾角标准差 σθ (degrees)', fontsize=11)
    ax3.set_ylabel('|ρhv|', fontsize=11)
    ax3.set_title('相关系数 |ρhv| 随倾角分散程度的变化', fontsize=12)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1.05)
    ax3.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    
    # 子图4: ZDR vs 倾角标准差
    ax4 = axes[1, 1]
    ax4.plot(sigma_range, ZDR_values, 'm-', linewidth=2)
    ax4.set_xlabel('倾角标准差 σθ (degrees)', fontsize=11)
    ax4.set_ylabel('ZDR (dB)', fontsize=11)
    ax4.set_title('差分反射率 ZDR 随倾角分散程度的变化', fontsize=12)
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(-5, 25)
    ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # 标记当前值
    ax3.axvline(x=sigma_theta, color='red', linestyle='--', alpha=0.7, label=f'当前 σθ={sigma_theta}°')
    ax4.axvline(x=sigma_theta, color='red', linestyle='--', alpha=0.7)
    
    cov_current = covariance_matrix_elements(D, wavelength, np.radians(sigma_theta))
    rho_current = calculate_rho_hv(cov_current['ShhSvv'], cov_current['ShhShh'], cov_current['SvvSvv'])
    ZDR_current = 10*np.log10(cov_current['ShhShh']/cov_current['SvvSvv'] + 1e-10)
    
    ax3.scatter([sigma_theta], [np.abs(rho_current)], color='red', s=100, zorder=5)
    ax4.scatter([sigma_theta], [ZDR_current], color='red', s=100, zorder=5)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 当前参数下的计算结果 ===")
    print(f"粒子直径: {D*1000:.1f} mm")
    print(f"波长: {wavelength*100:.1f} cm")
    print(f"倾角标准差: {sigma_theta}°")
    print(f"相关系数 |ρhv|: {np.abs(rho_current):.4f}")
    print(f"差分反射率 ZDR: {ZDR_current:.2f} dB")

In [ ]:
interact_orient = interact(plot_orientation_effects,
    D=widgets.FloatSlider(min=0.001, max=0.005, step=0.0005, value=0.002, 
                     description='粒子直径 (m)'),
    wavelength=widgets.FloatSlider(min=0.03, max=0.11, step=0.01, value=0.10, 
                                  description='波长 (m)'),
    sigma_theta=widgets.FloatSlider(min=1, max=60, step=1, value=15.0, 
                                   description='倾角标准差 (°)')
)

## 3. 雨滴 vs 冰晶的取向差异

In [ ]:
def compare_hydrometeors():
    """对比不同水凝物类型的取向特性"""
    D = 0.002  # 2mm
    wl = 0.107  # S-band
    
    # 不同水凝物类型的典型倾角标准差
    hydrometeors = {
        '小雨滴 (近球形)': 10,
        '大雨滴 (椭球)': 15,
        '雪晶 (平面)': 25,
        '冰晶 (柱状)': 35,
        '混合相粒子': 45
    }
    
    print("\n=== 不同水凝物类型的极化特征 ===")
    print(f"{'类型':<20} {'σθ (°)':<10} {'|ρhv|':<10} {'ZDR (dB)':<10}")
    print("-" * 50)
    
    for name, sigma in hydrometeors.items():
        cov = covariance_matrix_elements(D, wl, np.radians(sigma))
        rho = calculate_rho_hv(cov['ShhSvv'], cov['ShhShh'], cov['SvvSvv'])
        zdr = 10*np.log10(cov['ShhShh']/cov['SvvSvv'] + 1e-10)
        print(f"{name:<20} {sigma:<10.1f} {np.abs(rho):<10.4f} {zdr:<10.2f}")

In [ ]:
compare_hydrometeors()

## 练习题

1. **倾角分散效应**：当倾角标准差从5°增加到45°时，ρhv如何变化？

2. **ZDR变化**：取向高度一致的雨滴和取向随机的冰晶，哪个的ZDR更大？为什么？

3. **物理机制**：解释为什么倾角分散会导致ρhv降低？

4. **实际应用**：如何利用极化雷达数据推断降水粒子的取向特性？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 3, Springer.